In [ ]:
!pip install roboflow
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.5/84.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 95.2 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.11.0.86
    Uninstalling opencv-python-headless-4.11.0.86:
      Successfully uninstalled opencv-python-headless-4.11.0.86
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 949.8/949.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.2 MB/s eta 0:00:00

In [ ]:
import os
from roboflow import Roboflow
from google.colab import userdata
from ultralytics import YOLO
from ultralytics import RTDETR
import matplotlib.pyplot as plt
import cv2
import numpy as np
import shutil

#for UTF8 error when downloading ultralytics results from colab
import locale
locale.getpreferredencoding = lambda: "UTF-8"

#google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
def visualize_results(test_results, result_show_limit=20, txt_save_file=None):
  grid_imgs = []
  img_box_counts = []
  img_names = []
  for i, result in enumerate(test_results):
      if i >= result_show_limit:
          break
      # result.plot() returns an image (NumPy array) with predictions overlaid
      img = result.plot()
      grid_imgs.append(img)
      img_box_counts.append(len(result.boxes))
      img_names.append(result.path)

  cols = 4
  rows = int(np.ceil(len(grid_imgs) / cols))

  fig, axs = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))

  for idx, ax in enumerate(axs.flat):
      if idx < len(grid_imgs):

          ax.imshow(grid_imgs[idx])
          ax.set_title(f"Result {idx+1} with {img_box_counts[idx]} staves detected")
      ax.axis('off')

  if txt_save_file is not None:
      with open(txt_save_file, 'w') as f:
          for idx, img_name in enumerate(img_names):
              f.write(f"{img_box_counts[idx]} staves detected in image: {img_name}\n")

  plt.tight_layout()
  plt.show()

def get_staves_count_list(test_results):
  staves_count_list = []
  for result in test_results:
    staves_count_list.append(len(result.boxes))
  return staves_count_list
#
def save_results_to_indvidual_txt_files(test_results, output_folder, staves_threshold=500):
  os.makedirs(output_folder, exist_ok=True)
  os.makedirs(os.path.join(output_folder, "labels"), exist_ok=True)
  os.makedirs(os.path.join(output_folder, "images"), exist_ok=True)

  for i, result in enumerate(test_results):
    img_path = result.path
    img_name = os.path.basename(img_path)
    img_name_without_ext = os.path.splitext(img_name)[0]
    txt_file_path = os.path.join(output_folder, 'labels', f"{img_name_without_ext}.txt")

    conf_list = result.boxes.conf.cpu().numpy()
    box_list = result.boxes.xywhn
    cls_list = result.boxes.cls.cpu().numpy()

    box_count = len(conf_list)
    if box_count>=staves_threshold:
      shutil.copy(img_path, os.path.join(output_folder, "images", img_name))
      with open(txt_file_path, 'a') as result_file:
        for conf, box, cls in zip(conf_list, box_list, cls_list):
          if conf>0.1:
            result_file.write(f"{int(cls)} {box[0]} {box[1]} {box[2]} {box[3]}\n")

In [19]:
#weights_path = "/content/drive/MyDrive/Stave_Project/Developers/modelWeights/plankModel/detect/train2/weights/best.pt"
weights_path = "/content/drive/MyDrive/Stave_Project/Developers/modelWeights/runs/detect/train2/weights/last.pt"
input_folder = "/content/drive/MyDrive/Stave_Project/Developers/staveProjectImgsAllCropped"
output_folder = "/content/drive/MyDrive/Stave_Project/Developers/staveProjectImgsAllCroppedResults"

In [20]:
model = YOLO(weights_path)
test_results = []
for result in model.predict(source=input_folder, half=True, stream=True, max_det=1000):
  test_results.append(result)


image 1/696 /content/drive/MyDrive/Stave_Project/Developers/staveProjectImgsAllCropped/17289132322424750858174835555844 - Brandon Wagner_0.jpeg: 320x512 612 staves_regions, 11.6ms
image 2/696 /content/drive/MyDrive/Stave_Project/Developers/staveProjectImgsAllCropped/17289134066225452634665236915044 - Brandon Wagner_0.jpeg: 320x512 622 staves_regions, 11.1ms
image 3/696 /content/drive/MyDrive/Stave_Project/Developers/staveProjectImgsAllCropped/17289178627772986676454615492992 - Brandon Wagner_0.jpeg: 320x512 443 staves_regions, 11.1ms
image 4/696 /content/drive/MyDrive/Stave_Project/Developers/staveProjectImgsAllCropped/1728932532448232548761055507990 - Brandon Wagner_0.jpeg: 288x512 310 staves_regions, 11.0ms
image 5/696 /content/drive/MyDrive/Stave_Project/Developers/staveProjectImgsAllCropped/17289325762097634202431665002761 - Brandon Wagner_0.jpeg: 320x512 529 staves_regions, 11.9ms
image 6/696 /content/drive/MyDrive/Stave_Project/Developers/staveProjectImgsAllCropped/1728932752409

In [ ]:
visualize_results(test_results, result_show_limit=700)
#save_results_to_indvidual_txt_files(test_results, output_folder)

In [35]:
gt_staves_count = [
    609, 655, 624, 645, 628, 689, 809, 725, 659, 698, 645, 643, 711, 657, 662, 658, 667, 670, 694, 786, 786, 664, 710,
    647, 678, 641, 648, 695, 614, 653, 667, 646, 648, 627, 667, 669, 697, 633, 752, 775, 630, 624, 725, 613, 624, 719,
    617, 538, 615, 675, 651, 668, 683, 614, 677, 677, 668, 683, 680, 745, 643, 580, 580, 699, 1214, 638, 624, 680, 665,
    713, 713, 628, 628, 696, 655, 693, 725, 675, 688, 692, 769, 661, 627, 692, 693, 640, 621, 612, 660, 649, 661, 649,
    729, 729, 659, 680, 670, 634, 682, 654, 643, 654, 663, 665, 634, 654, 662, 662, 629, 689, 668, 718, 680, 758, 685,
    630, 654, 679, 654, 644, 704, 671, 690, 646, 646, 1348, 629, 670, 678, 635, 594, 632, 745, 725, 695, 696, 668, 711,
    660, 636, 833, 833, 657, 643, 662, 714, 643, 711, 843, 649, 732, 687, 688, 685, 672, 673, 625, 773, 657, 730, 656,
    691, 691, 655, 658, 668, 670, 632, 651, 655, 693, 663, 695, 692, 686, 650, 675, 746, 680, 764, 652, 657, 660, 631,
    730, 712, 680, 666, 699, 676, 689, 686, 808, 673, 653, 639, 639, 746, 684, 742, 659, 730, 742, 659, 730, 667, 667,
    698, 772, 772, 716, 605, 605, 716, 698, 681, 675, 681, 675, 652, 653, 701, 666, 684, 710, 676, 652, 687, 675, 689,
    688, 637, 672, 700, 801, 647, 669, 706, 672, 798, 734, 734, 654, 673, 668, 692, 667, 683, 673, 772, 646, 793, 668,
    671, 683, 668, 702, 659, 636, 671, 762, 665, 657, 759, 646, 770, 747, 707, 758, 778, 646, 620, 664, 711, 658, 643,
    638, 646, 658, 664, 655, 662, 734, 679, 712, 725, 714, 675, 695, 656, 695, 694, 775, 668, 657, 659, 730, 666, 739,
    623, 670, 689, 658, 728, 655, 638, 682, 724, 690, 355, 693, 650, 674, 665, 754, 635, 627, 640, 622, 738, 620, 635,
    637, 747, 666, 635, 621, 653, 709, 640, 621, 705, 623, 670, 685, 625, 661, 613, 623, 652, 700, 622, 618, 691, 631,
    658, 719, 642, 691, 660, 722, 662, 696, 641, 658, 674, 694, 709, 705, 700, 706, 703, 655, 620, 651, 649, 619, 647,
    654, 745, 614, 636, 622, 705, 653, 650, 709, 632, 617, 630, 690, 644, 613, 639, 641, 659, 642, 630, 711, 666, 648,
    708, 644, 670, 630, 710, 633, 762, 660, 637, 648, 665, 677, 645, 642, 768, 715, 656, 649, 642, 655, 649, 679, 669,
    648, 657, 728, 657, 652, 646, 663, 661, 662, 617, 657, 689, 656, 760, 666, 712, 643, 680, 651, 651, 651, 661, 686,
    687, 745, 708, 664, 642, 625, 704, 618, 676, 669, 709, 655, 626, 674, 671, 630, 647, 660, 703, 628, 611, 611, 629,
    646, 731, 634, 623, 653, 647, 638, 703, 669, 619, 654, 718, 673, 687, 633, 648, 637, 646, 751, 637, 677, 677, 742,
    649, 652, 667, 647, 629, 772, 665, 751, 648, 672, 671, 678, 629, 630, 636, 651, 684, 653, 638, 650, 645, 732, 646,
    621, 642, 620, 653, 622, 609, 693, 662, 650, 640, 636, 639, 658, 645, 657, 727, 684, 659, 670, 635, 669, 690, 716,
    640, 731, 676, 701, 646, 682, 679, 686, 717, 641, 689, 631, 717, 655, 646, 638, 702, 700, 688, 655, 644, 691, 644,
    618, 682, 656, 689, 656, 677, 670, 666, 617, 643, 627, 710, 699, 660, 643, 716, 624, 696, 659, 631, 638, 644, 740,
    674, 674, 678, 651, 628, 636, 622, 679, 652, 629, 631, 680, 642, 631, 634, 667, 634, 611, 732, 716, 673, 717, 640,
    678, 773, 663, 621, 620, 605, 619, 644, 629, 648, 645, 608, 732, 624, 634, 610, 660, 660, 620, 637, 633, 626, 608,
    631, 640, 690, 719, 650, 750, 750, 647, 640, 640, 647, 634, 634, 648, 623, 652, 629, 667, 690, 648, 675, 675, 660,
    694, 694, 646, 646, 721, 721, 641, 631, 660, 648, 636, 653, 639, 788, 628, 665, 623, 653, 701, 701, 680, 627, 662,
    673, 673, 860, 860, 650, 662, 678, 798, 798, 645, 753, 753, 635, 685, 687, 659, 659, 670, 675, 650, 743, 738, 760,
    714, 719, 719, 704, 686, 676

]
pred_stave_count = get_staves_count_list(test_results)

#calc mse
mse = np.mean((np.array(gt_staves_count) - np.array(pred_stave_count))**2)
print(f"MSE: {mse}")

#calc mean accuracy
mean_accuracy = (np.array(gt_staves_count)-np.array(pred_stave_count))/np.array(gt_staves_count)
mean_accuracy = np.mean(mean_accuracy)
print(f"Mean Accuracy: {mean_accuracy}")

MSE: 92028.47701149425
Mean Accuracy: 0.37355009151654167
